In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/09/23 00:44:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
spark.sql("""
SHOW DATABASES
""").show()

+------------+
|   namespace|
+------------+
|     default|
|transform_db|
+------------+



In [5]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS transform_db
LOCATION 's3a://crypto-data-lake/transform_zone/'
""")

DataFrame[]

In [7]:
schema = types.StructType([
    types.StructField("agg_trade_id", types.LongType(), True),
    types.StructField("price", types.DoubleType(), True),
    types.StructField("quantity", types.DoubleType(), True),
    types.StructField("timestamp", types.LongType(), True),
    types.StructField("symbol", types.StringType(), True)
])

In [8]:
data = [
    (1, 50000.0, 0.1, 1695360000, "BTCUSDT"),
    (2, 50010.0, 0.2, 1695363600, "BTCUSDT")
]

In [21]:
df = spark.createDataFrame(data, schema)

In [10]:
df.writeTo("transform_db.aggTrades").tableProperty("format-version", "2").create()

25/09/22 12:45:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [7]:
spark.sql("SHOW TABLES IN transform_db").show()

+------------+----------+-----------+
|   namespace| tableName|isTemporary|
+------------+----------+-----------+
|transform_db| aggtrades|      false|
|transform_db|test_table|      false|
|transform_db|aggtrades2|      false|
+------------+----------+-----------+



In [14]:
spark.sql("""
CREATE TABLE IF NOT EXISTS transform_db.test_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
)
USING iceberg
LOCATION 's3a://crypto-data-lake/transform_zone/test_table'
""")

DataFrame[]

In [11]:
# Insert sample data
spark.sql("""
INSERT INTO transform_db.test_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP),
    (3, 'Charlie', CURRENT_TIMESTAMP)
""")

DataFrame[]

In [12]:
spark.sql("""
select count(*) from transform_db.test_table
""").show()

+--------+
|count(1)|
+--------+
|       9|
+--------+



In [19]:
# Query the table
result = spark.sql("SELECT * FROM transform_db.test_table")
result.show()

+---+-------+--------------------+
| id|   name|          created_at|
+---+-------+--------------------+
|  1|  Alice|2025-09-22 12:57:...|
|  2|    Bob|2025-09-22 12:57:...|
|  3|Charlie|2025-09-22 12:57:...|
+---+-------+--------------------+



In [22]:
df.writeTo("transform_db.aggTrades2").tableProperty("format-version", "2").create()

In [13]:
spark.sql("SELECT * FROM transform_db.test_table.snapshots").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-09-22 12:57:...|8577955136122491824|               NULL|   append|s3a://crypto-data...|{spark.app.id -> ...|
|2025-09-22 13:06:...|6591235640726722239|8577955136122491824|   append|s3a://crypto-data...|{spark.app.id -> ...|
|2025-09-22 13:07:...|7452194205708839951|6591235640726722239|   append|s3a://crypto-data...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [84]:
spark.sql("SHOW DATABASES").show()

+------------+
|   namespace|
+------------+
|     default|
|     test_db|
|transform_db|
+------------+



In [4]:
from datetime import datetime

In [61]:
current_date = datetime.now().strftime("%Y-%m-%d")
print(current_date)

2025-09-23


In [59]:
data = [
    (1, "Tu"),
    (2, "Tuan"),
    (3, "Tien")
]
schema = types.StructType([
    types.StructField('user_id', types.LongType(), True), 
    types.StructField('name', types.StringType(), True)
])

In [60]:
df = spark.createDataFrame(data, schema)

In [62]:
df = df.withColumn("ingest_date", F.to_date(F.lit(current_date), "yyyy-MM-dd"))

In [63]:
df.show()

+-------+----+-----------+
|user_id|name|ingest_date|
+-------+----+-----------+
|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|
|      3|Tien| 2025-09-23|
+-------+----+-----------+



In [64]:
df.writeTo("transform_db.users").tableProperty("format-version", "2").partitionedBy("ingest_date").createOrReplace()

In [93]:
resutl = spark.sql("""
select * from transform_db.users
""")

+-------+----+-----------+
|user_id|name|ingest_date|
+-------+----+-----------+
|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|
|      3|Tien| 2025-09-23|
+-------+----+-----------+



In [75]:
print(type(df), type(resutl))

<class 'pyspark.sql.dataframe.DataFrame'> <class 'pyspark.sql.dataframe.DataFrame'>


In [83]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS test_db
LOCATION 's3a://crypto-data-lake/test_zone/'
""")

DataFrame[]

In [89]:
df.writeTo("test_db.users").tableProperty("format-version", "2").partitionedBy("ingest_date").append()

In [92]:
spark.sql("""
select * from test_db.users.snapshots
""").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-09-23 01:34:...| 245542817659610434|               NULL|   append|s3a://crypto-data...|{spark.app.id -> ...|
|2025-09-23 01:39:...|6232087004061595648|               NULL|   append|s3a://crypto-data...|{spark.app.id -> ...|
|2025-09-23 01:40:...|2506971100231471199|6232087004061595648|   append|s3a://crypto-data...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [95]:
spark.sql("""
select * from transform_db.users
""").show()

+-------+----+-----------+
|user_id|name|ingest_date|
+-------+----+-----------+
|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|
|      3|Tien| 2025-09-23|
+-------+----+-----------+



In [96]:
spark.sql("""
select * from test_db.users
""").show()

+-------+----+-----------+
|user_id|name|ingest_date|
+-------+----+-----------+
|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|
|      3|Tien| 2025-09-23|
|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|
|      3|Tien| 2025-09-23|
+-------+----+-----------+



In [99]:
spark.sql("""
select * 
from
    test_db.users tu
    JOIN transform_db.users mu ON tu.user_id != mu.user_id
""").show()

+-------+----+-----------+-------+----+-----------+
|user_id|name|ingest_date|user_id|name|ingest_date|
+-------+----+-----------+-------+----+-----------+
|      1|  Tu| 2025-09-23|      2|Tuan| 2025-09-23|
|      1|  Tu| 2025-09-23|      3|Tien| 2025-09-23|
|      2|Tuan| 2025-09-23|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|      3|Tien| 2025-09-23|
|      3|Tien| 2025-09-23|      1|  Tu| 2025-09-23|
|      3|Tien| 2025-09-23|      2|Tuan| 2025-09-23|
|      1|  Tu| 2025-09-23|      2|Tuan| 2025-09-23|
|      1|  Tu| 2025-09-23|      3|Tien| 2025-09-23|
|      2|Tuan| 2025-09-23|      1|  Tu| 2025-09-23|
|      2|Tuan| 2025-09-23|      3|Tien| 2025-09-23|
|      3|Tien| 2025-09-23|      1|  Tu| 2025-09-23|
|      3|Tien| 2025-09-23|      2|Tuan| 2025-09-23|
+-------+----+-----------+-------+----+-----------+

